# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

**Unit of analysis**

One row represents the daily SEO performance of one webpage (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).

**Time window**

The warehouse stores daily observations. The `month` column identifies the reporting month.


In [1]:
import pyarrow.parquet as pq
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

table = pq.read_table(filename)
df = table.to_pandas()

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date Range:", df["report_date"].min(), "to", df["report_date"].max())

display(df[["report_date","client_hash_id","content_hash_id"]].head())


Saving fact_content_daily_performance_sample (1).parquet to fact_content_daily_performance_sample (1).parquet
Rows: 11694072
Columns: 31
Date Range: 2026-06-01 to 2026-06-30


,report_date,client_hash_id,content_hash_id
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9



## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- scroll_events

### Label / Proxy
Refresh Priority Score (constructed later from SEO signals).

### Context
- report_date
- month
- client_hash_id
- content_hash_id

### Excluded
- client_hash_id
- content_hash_id

These are identifiers and are not predictive features.


In [2]:

print("All Columns")
for i,col in enumerate(df.columns,1):
    print(f"{i}. {col}")


All Columns
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events
31. month



## 3. Verify it with queries (grain, counts, missing values, windows)

These queries verify the contract before modeling.


In [3]:

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique Clients:", df["client_hash_id"].nunique())
print("Unique Pages:", df["content_hash_id"].nunique())
print("Months:", df["month"].unique())
print()

print("Missing Values")
display(df.isnull().sum())

print("GSC availability")
display(df["gsc_data_available"].value_counts())

print("GA4 availability")
display(df["ga4_data_available"].value_counts())


Rows: 11694072
Columns: 31
Unique Clients: 65
Unique Pages: 409205
Months: ['2026-06']

Missing Values


,0
report_date,0
client_hash_id,0
content_hash_id,0
client_has_gsc,0
client_has_ga4,0
gsc_data_available,0
ga4_data_available,2397428
gsc_impressions,0
gsc_clicks,0
gsc_sum_position,4


GSC availability


,count
gsc_data_available,
False,7815135
True,3878937


GA4 availability


,count
ga4_data_available,
False,8651918
True,644726



## 4. Data limits

- This is a warehouse sample, not the complete production warehouse.
- Historical observations cannot guarantee future rankings.
- Missing GA4/GSC values exist for some pages.
- Hash IDs identify entities but contain no business meaning.
- The data supports decisions but does not prove causation.


In [4]:

missing = df.isnull().sum()
missing = missing[missing>0]
print("Columns with missing values:")
display(missing)


Columns with missing values:


,0
ga4_data_available,2397428
gsc_sum_position,4
gsc_avg_position,7815170
ga4_pageviews,2397428
ga4_sessions,2397428
ga4_users,2397428
ga4_engaged_sessions,2397428
ga4_total_engagement_sec,2397428
sessions_organic,2397428
sessions_direct,2397428



# Self-check

- ✅ Unit of analysis defined
- ✅ Time window verified
- ✅ Features documented
- ✅ Label/proxy documented
- ✅ Context fields documented
- ✅ Excluded fields documented
- ✅ Verification queries included
- ✅ Data limits explained
- ✅ Notebook runs from top to bottom
